First i download the backup file, that Pair 1 comepleted which was Mehtab, and Sadia, and then i restored on my system and completed my part, and handed it down to my teammate Saira.

In [9]:
-- 1: ================================================================
-- Author:      Mst Rahi
-- Procedure:   Testing foreign key integrity and schema relationships
-- Create date: 04/13/25
-- Description: To test the integrity of foreign keys and schema relationships 

-- ================================================================

USE [G10_2];
GO


Commands completed successfully.

Total execution time: 00:00:00.007

In [10]:
--2: ================================================================
-- Author:      Mst Rahi
-- Procedure:   usp_TrackWorkFlow
-- Create date: 04/12/2025
-- Description: Inserts a new workflow step with the current timestamp for a given user authorization key and description.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[usp_TrackWorkFlow]
    @UserAuthKey INT,
    @WorkflowStepDescription NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO [dbo].[WorkflowSteps]
    (
        UserAuthorizationKey,       -- Must match the actual column name in your table
        WorkflowStepDescription,
        StartingDateTime
    )
    VALUES
    (
        @UserAuthKey,
        @WorkflowStepDescription,
        GETDATE()
    );
END;
GO


Commands completed successfully.

Total execution time: 00:00:00.055

In [3]:
--3: ================================================================
-- Author:      Mst Rahi
-- Procedure:   usp_TrackWorkFlow
-- Create date: 04/12/2025
-- Description: Inserts a new workflow step with the current timestamp into the Process.WorkflowSteps table for a given user authorization key and step description.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[usp_TrackWorkFlow]
    @UserAuthKey INT,
    @WorkflowStepDescription NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO [Process].[WorkflowSteps] 
    (
        UserAuthorizationKey,      
        WorkflowStepDescription,
        StartingDateTime
    )
    VALUES
    (
        @UserAuthKey,
        @WorkflowStepDescription,
        GETDATE()
    );
END;
GO


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.081

In [15]:
-- --4: ================================================================
-- Author:      Mst Rahi
-- Procedure:   usp_TrackWorkFlow
-- Create date: 04/12/2025
-- Description: Inserts a workflow step with the provided description 
--              and current timestamp into the Process.WorkflowSteps 
--              table for the specified user authorization key.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[usp_TrackWorkFlow]
    @UserAuthKey INT,
    @WorkflowStepDescription NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO [Process].[WorkflowSteps]  -- or [dbo].[WorkflowSteps], whichever is correct
    (
        UserAuthorizationKey,
        WorkflowStepDescription,
        StartingDateTime
    )
    VALUES
    (
        @UserAuthKey,
        @WorkflowStepDescription,
        GETDATE()
    );
END;
GO


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.014

In [17]:
-- 5:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   usp_TrackWorkFlow
-- Create date: 04/12/2025
-- Description: Creates a sequence for generating WorkflowSteps keys 
--              starting from 1 and incrementing by 1.
-- ================================================================
CREATE SEQUENCE [Process].[WorkflowStepsKeySeq]
    START WITH 1
    INCREMENT BY 1;
GO


Commands completed successfully.

Total execution time: 00:00:00.027

In [4]:
-- 6:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   QueryPrimaryKeyInfo
-- Create date: 04/12/2025
-- Description: Retrieves the primary key of Process.WorkflowSteps.
-- ================================================================
USE [G10_2];
GO

SELECT CONSTRAINT_NAME, TABLE_SCHEMA, TABLE_NAME, CONSTRAINT_TYPE
FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
WHERE TABLE_SCHEMA = 'Process'
  AND TABLE_NAME = 'WorkflowSteps'
  AND CONSTRAINT_TYPE = 'PRIMARY KEY';


Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.184

CONSTRAINT_NAME,TABLE_SCHEMA,TABLE_NAME,CONSTRAINT_TYPE
PK_WorkFlowSteps_WorkFlowStepsKey,Process,WorkflowSteps,PRIMARY KEY


In [22]:
-- 7:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   DropPKAndColumn_WorkflowSteps
-- Create date: 04/12/2025
-- Description: Drops the primary key and the column WorkFlowStepsKey 
--              from Process.WorkflowSteps if they exist.
-- ================================================================
USE [G10_2];
GO

-- Drop the primary key constraint if it exists.
IF EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS 
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY'
)
BEGIN
    DECLARE @PKName NVARCHAR(128);
    SELECT @PKName = CONSTRAINT_NAME 
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS 
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY';

    PRINT 'Dropping primary key constraint: ' + @PKName;
    EXEC('ALTER TABLE [Process].[WorkflowSteps] DROP CONSTRAINT [' + @PKName + ']');
END
ELSE
BEGIN
    PRINT 'No primary key constraint found on Process.WorkflowSteps.';
END
GO

-- Drop the column WorkFlowStepsKey if it exists.
IF EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND COLUMN_NAME = 'WorkFlowStepsKey'
)
BEGIN
    PRINT 'Dropping column WorkFlowStepsKey.';
    ALTER TABLE [Process].[WorkflowSteps] DROP COLUMN [WorkFlowStepsKey];
END
GO


Commands completed successfully.

No primary key constraint found on Process.WorkflowSteps.

Commands completed successfully.

Total execution time: 00:00:00.133

In [23]:
-- 8:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   AddIdentityAndPrimaryKey
-- Create date: 04/12/2025
-- Description: Adds WorkFlowStepsKey as an identity column and 
--              sets it as the primary key in Process.WorkflowSteps.
-- ================================================================
-- Add the column as an identity column.
ALTER TABLE [Process].[WorkflowSteps]
ADD [WorkFlowStepsKey] INT IDENTITY(1,1) NOT NULL;
GO

-- Add the primary key constraint.
ALTER TABLE [Process].[WorkflowSteps]
ADD CONSTRAINT [PK_WorkflowSteps_WorkFlowStepsKey] PRIMARY KEY ([WorkFlowStepsKey]);
GO


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.035

In [60]:
-- 9:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   RebuildPKAndAddIdentity
-- Create date: 04/12/2025
-- Description: Drops the existing primary key and WorkFlowStepsKey column, 
--              adds WorkFlowStepsKey as an identity column, 
--              and creates a new primary key on it.
-- ================================================================
USE [G10_2];
GO

--------------------------------------------------------------------------------
-- 1) Drop the PK constraint if it exists
--------------------------------------------------------------------------------
IF EXISTS (
    SELECT * 
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY'
)
BEGIN
    DECLARE @pkName NVARCHAR(128);

    SELECT @pkName = CONSTRAINT_NAME
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY';

    EXEC('ALTER TABLE [Process].[WorkflowSteps] DROP CONSTRAINT [' + @pkName + ']');
END;

--------------------------------------------------------------------------------
-- 2) Drop the WorkFlowStepsKey column (assuming no critical data to preserve)
--------------------------------------------------------------------------------
IF EXISTS (
    SELECT * 
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND COLUMN_NAME = 'WorkFlowStepsKey'
)
BEGIN
    ALTER TABLE [Process].[WorkflowSteps]
    DROP COLUMN [WorkFlowStepsKey];
END;

--------------------------------------------------------------------------------
-- 3) Add WorkFlowStepsKey as an IDENTITY
--------------------------------------------------------------------------------
ALTER TABLE [Process].[WorkflowSteps]
ADD [WorkFlowStepsKey] INT IDENTITY(1,1) NOT NULL;

--------------------------------------------------------------------------------
-- 4) Create a primary key constraint on that new column
--------------------------------------------------------------------------------
ALTER TABLE [Process].[WorkflowSteps]
ADD CONSTRAINT [PK_WorkflowSteps_WorkFlowStepsKey] 
PRIMARY KEY ([WorkFlowStepsKey]);
GO


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.164

In [61]:
-- 10:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   InsertAndSelectWorkflowStep
-- Create date: 04/12/2025
-- Description: Inserts a test row into Process.WorkflowSteps and 
--              selects the top 10 rows ordered by WorkFlowStepsKey.
-- ================================================================
INSERT INTO [Process].[WorkflowSteps]
(
    WorkflowStepDescription,
    WorkFlowStepTableRowCount,
    StartingDateTime,
    EndingDateTime,
    ClassTime,
    UserAuthorizationKey
)
VALUES
(
    'First test row',
    0,
    GETDATE(),
    NULL,
    '10:45',
    1
);

SELECT TOP (10) *
FROM [Process].[WorkflowSteps]
ORDER BY WorkFlowStepsKey DESC;


The statement has been terminated.

(0 rows affected)

Total execution time: 00:00:00.038

WorkFlowStepKey,WorkFlowStepDescription,WorkFlowStepTableRowCount,StartingDateTime,EndingDateTime,ClassTime,UserAuthorizationKey,WorkFlowStepsKey


: Msg 515, Level 16, State 2, Line 1
Cannot insert the value NULL into column 'WorkFlowStepKey', table 'G10_2.Process.WorkflowSteps'; column does not allow nulls. INSERT fails.

In [5]:
-- 11:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   FullWorkflowStepsRebuild
-- Create date: 04/12/2025
-- Description: Drops the existing primary key, unwanted columns, 
--              re-creates an identity column, and tests the workflow 
--              steps insertion process.
-- ================================================================
USE [G10_2];
GO

---------------------------------------------------------------------
-- Step 1: Drop the existing primary key constraint (if any)
---------------------------------------------------------------------
IF EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY'
)
BEGIN
    DECLARE @PKName NVARCHAR(128);
    SELECT @PKName = CONSTRAINT_NAME
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND CONSTRAINT_TYPE = 'PRIMARY KEY';
    
    PRINT 'Dropping primary key constraint: ' + @PKName;
    EXEC('ALTER TABLE [Process].[WorkflowSteps] DROP CONSTRAINT [' + @PKName + ']');
END
ELSE
BEGIN
    PRINT 'No primary key constraint found on [Process].[WorkflowSteps].';
END;
GO

---------------------------------------------------------------------
-- Step 2: Drop the unwanted key column (singular name)
--         This is the column that is not auto‐generated.
---------------------------------------------------------------------
IF EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND COLUMN_NAME = 'WorkFlowStepKey'
)
BEGIN
    PRINT 'Dropping column WorkFlowStepKey...';
    ALTER TABLE [Process].[WorkflowSteps]
    DROP COLUMN [WorkFlowStepKey];
END
ELSE
BEGIN
    PRINT 'Column WorkFlowStepKey does not exist, skipping drop.';
END;
GO

---------------------------------------------------------------------
-- Step 3: Drop the existing (possibly incorrectly defined) column 
--         that should be our identity primary key.
--         We want to re-create WorkFlowStepsKey (plural) as the sole key.
---------------------------------------------------------------------
IF EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Process'
      AND TABLE_NAME = 'WorkflowSteps'
      AND COLUMN_NAME = 'WorkFlowStepsKey'
)
BEGIN
    PRINT 'Dropping column WorkFlowStepsKey for re-creation...';
    ALTER TABLE [Process].[WorkflowSteps]
    DROP COLUMN [WorkFlowStepsKey];
END
ELSE
BEGIN
    PRINT 'Column WorkFlowStepsKey does not exist, will be created.';
END;
GO

---------------------------------------------------------------------
-- Step 4: Re-add the WorkFlowStepsKey column as an IDENTITY column.
---------------------------------------------------------------------
ALTER TABLE [Process].[WorkflowSteps]
ADD [WorkFlowStepsKey] INT IDENTITY(1,1) NOT NULL;
GO

---------------------------------------------------------------------
-- Step 5: Re-create the primary key constraint on the new identity column.
---------------------------------------------------------------------
ALTER TABLE [Process].[WorkflowSteps]
ADD CONSTRAINT [PK_WorkFlowSteps_WorkFlowStepsKey] 
PRIMARY KEY ([WorkFlowStepsKey]);
GO

---------------------------------------------------------------------
-- Step 6: Create (or update) the stored procedure to log workflow steps.
--         Note: It does not supply a value for the identity column.
---------------------------------------------------------------------
CREATE OR ALTER PROCEDURE [dbo].[usp_TrackWorkFlow]
    @UserAuthKey INT,
    @WorkflowStepDescription NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO [Process].[WorkflowSteps]
    (
        UserAuthorizationKey,
        WorkflowStepDescription,
        StartingDateTime
    )
    VALUES
    (
        @UserAuthKey,
        @WorkflowStepDescription,
        GETDATE()
    );
END;
GO

---------------------------------------------------------------------
-- Step 7: Test a direct INSERT (without the stored procedure)
---------------------------------------------------------------------
INSERT INTO [Process].[WorkflowSteps]
(
    WorkflowStepDescription,
    WorkFlowStepTableRowCount,
    StartingDateTime,
    EndingDateTime,
    ClassTime,
    UserAuthorizationKey
)
VALUES
(
    'Test direct insert row',
    0,
    GETDATE(),
    NULL,
    '10:45',
    1
);
GO

---------------------------------------------------------------------
-- Step 8: Test the stored procedure
---------------------------------------------------------------------
EXEC [dbo].[usp_TrackWorkFlow]
     @UserAuthKey = 2,
     @WorkflowStepDescription = 'Test via stored procedure';
GO

---------------------------------------------------------------------
-- Step 9: Query the table to verify the new rows.
---------------------------------------------------------------------
SELECT TOP (10) *
FROM [Process].[WorkflowSteps]
ORDER BY WorkFlowStepsKey DESC;
GO


Commands completed successfully.

Dropping primary key constraint: PK_WorkFlowSteps_WorkFlowStepsKey

Column WorkFlowStepKey does not exist, skipping drop.

Dropping column WorkFlowStepsKey for re-creation...

Commands completed successfully.

...

(1 row affected)

Commands completed successfully.

(10 rows affected)

Total execution time: 00:00:00.311

WorkFlowStepDescription,WorkFlowStepTableRowCount,StartingDateTime,EndingDateTime,ClassTime,UserAuthorizationKey,WorkFlowStepsKey
Test via stored procedure,0,2025-04-14 00:05:18.9433333,2025-04-14 00:05:18.9500000,10:45,2,11
Test direct insert row,0,2025-04-14 00:05:18.9400000,NULL,10:45,1,10
ETL process completed,0,2025-04-13 22:36:39.0633333,2025-04-13 22:36:39.0666666,10:45,1,9
Starting load for Data table,0,2025-04-13 22:36:38.9833333,2025-04-13 22:36:38.9866666,10:45,1,8
ETL process started,0,2025-04-13 22:36:38.9800000,2025-04-13 22:36:38.9833333,10:45,1,7
ETL process completed,0,2025-04-13 04:29:43.5400000,2025-04-13 04:29:43.5266666,10:45,1,6
Starting load for Data table,0,2025-04-13 04:29:43.5333333,2025-04-13 04:29:43.5233333,10:45,1,5
ETL process started,0,2025-04-13 04:29:43.5300000,2025-04-13 04:29:43.5200000,10:45,1,4
Test workflow log entry,0,2025-04-13 04:26:29.3000000,2025-04-13 04:26:29.3100000,10:45,1,3
Test via stored procedure,0,2025-04-13 04:25:10.2466667,2025-04-13 04:25:10.2433333,10:45,2,2


In [68]:
-- 12:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   sp_DropAllForeignKeys
-- Create date: 04/12/2025
-- Description: Drops all foreign key constraints in the database.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[sp_DropAllForeignKeys]
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @sql NVARCHAR(MAX) = N'';

    -- Build a list of DROP CONSTRAINT statements for all foreign keys.
    SELECT @sql += N'ALTER TABLE ' 
                    + QUOTENAME(OBJECT_SCHEMA_NAME(parent_object_id)) + N'.' 
                    + QUOTENAME(OBJECT_NAME(parent_object_id)) 
                    + N' DROP CONSTRAINT ' + QUOTENAME(name) + N';' + CHAR(13)
    FROM sys.foreign_keys;

    -- For debugging, you can print the generated SQL.
    PRINT @sql;

    EXEC sp_executesql @sql;
END;
GO

-- Test it (if needed):
-- EXEC [dbo].[sp_DropAllForeignKeys];


Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.023

In [69]:
-- 13:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   sp_TruncateTableData
-- Create date: 04/12/2025
-- Description: Truncates data from a specified table, with a check to 
--              prevent truncating certain tables like 'FileUpload'.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[sp_TruncateTableData]
    @SchemaName NVARCHAR(128),
    @TableName NVARCHAR(128)
AS
BEGIN
    SET NOCOUNT ON;

    -- Prevent truncating certain tables (e.g., 'FileUpload').
    IF @TableName = 'FileUpload'
    BEGIN
        RAISERROR('Cannot truncate the FileUpload table.', 16, 1);
        RETURN;
    END

    DECLARE @sql NVARCHAR(MAX);
    SET @sql = N'TRUNCATE TABLE ' + QUOTENAME(@SchemaName) + N'.' + QUOTENAME(@TableName) + N';';

    PRINT @sql;  -- For debugging.
    EXEC sp_executesql @sql;
END;
GO

-- Test it (replace 'YourTestTable' with an actual table name for testing):
-- EXEC [dbo].[sp_TruncateTableData] @SchemaName = 'dbo', @TableName = 'YourTestTable';


Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.029

In [70]:
-- 14:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   GetRowCount
-- Create date: 04/12/2025
-- Description: Returns the row count of a specified table in a given schema.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER FUNCTION [dbo].[GetRowCount]
(
    @SchemaName NVARCHAR(128), 
    @TableName NVARCHAR(128)
)
RETURNS INT
AS
BEGIN
    DECLARE @sql NVARCHAR(MAX);
    DECLARE @count INT;
    SET @sql = N'SELECT @cnt = COUNT(*) FROM ' 
               + QUOTENAME(@SchemaName) + N'.' + QUOTENAME(@TableName) + N';';

    EXEC sp_executesql @sql, N'@cnt INT OUTPUT', @cnt = @count OUTPUT;
    RETURN @count;
END;
GO

-- Test it:
-- SELECT dbo.GetRowCount('dbo', 'YourTestTable') AS RowCount;


Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.022

In [1]:
-- 15:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   usp_LoadFactData
-- Create date: 04/12/2025
-- Description: Loads data into the FactData table, handling workflow logging, 
--              truncation of existing data, and row count checks before and after the load.
-- ================================================================
USE [G10_2];
GO

CREATE OR ALTER PROCEDURE [dbo].[usp_LoadFactData]
    @UserAuthKey INT
AS
BEGIN
    SET NOCOUNT ON;

    -------------------------------------------------------------------
    -- 1) Log the start of the workflow.
    -------------------------------------------------------------------
    EXEC [dbo].[usp_TrackWorkFlow]
         @UserAuthKey, 
         N'Starting FactData load';

    -------------------------------------------------------------------
    -- 2) Check row count before truncation.
    -------------------------------------------------------------------
    DECLARE @RowCountBefore INT = dbo.GetRowCount('dbo', 'FactData');
    PRINT 'FactData row count BEFORE truncation: ' + CAST(@RowCountBefore AS NVARCHAR(10));

    -------------------------------------------------------------------
    -- 3) Truncate the FactData table.
    -------------------------------------------------------------------
    EXEC [dbo].[sp_TruncateTableData] @SchemaName = 'dbo', @TableName = 'FactData';

    DECLARE @RowCountAfterTruncate INT = dbo.GetRowCount('dbo', 'FactData');
    PRINT 'FactData row count AFTER truncation: ' + CAST(@RowCountAfterTruncate AS NVARCHAR(10));

    -------------------------------------------------------------------
    -- 4) Load new data into FactData.
    --
    -- This example assumes you load data from [dbo].[StagingSalesData]
    -- by joining with dimension tables for surrogate keys.
    --
    -- Adjust the column list and JOIN conditions as needed.
    -------------------------------------------------------------------
    DECLARE @sql NVARCHAR(MAX);
    SET @sql = N'
        INSERT INTO dbo.FactData
        (
            -- If FactData uses an IDENTITY for its PK, omit the PK column here.
            DimCustomerKey,
            DimProductKey,
            DimProductCategoryKey,
            DimProductSubcategoryKey,
            -- other fact columns,
            UserAuthorizationKey,
            DateAdded,
            DateLastUpdated
        )
        SELECT
            c.CustomerKey,
            p.ProductKey,
            pc.ProductCategoryKey,
            psc.ProductSubcategoryKey,
            -- other columns,
            ' + CAST(@UserAuthKey AS NVARCHAR(10)) + N',
            GETDATE(),
            GETDATE()
        FROM dbo.StagingSalesData st
        INNER JOIN dbo.DimCustomer c ON st.CustomerAlternateKey = c.CustomerAlternateKey
        INNER JOIN dbo.DimProduct p ON st.ProductAlternateKey = p.ProductAlternateKey
        INNER JOIN dbo.DimProductCategory pc ON st.ProductCategoryName = pc.ProductCategoryName
        INNER JOIN dbo.DimProductSubcategory psc ON st.ProductSubcategoryName = psc.ProductSubcategoryName;
    ';

    PRINT @sql;  -- For debugging purposes.
    EXEC sp_executesql @sql;

    -------------------------------------------------------------------
    -- 5) Check row count after load.
    -------------------------------------------------------------------
    DECLARE @RowCountAfterLoad INT = dbo.GetRowCount('dbo', 'FactData');
    PRINT 'FactData row count AFTER load: ' + CAST(@RowCountAfterLoad AS NVARCHAR(10));

    -------------------------------------------------------------------
    -- 6) Log the end of the workflow.
    -------------------------------------------------------------------
    EXEC [dbo].[usp_TrackWorkFlow]
         @UserAuthKey,
         N'Completed FactData load';
END;
GO

-- Test the load procedure (adjust the user key as necessary):
-- EXEC [dbo].[usp_LoadFactData] @UserAuthKey = 1;


Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.041

In [72]:
-- 16:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   Test workflow log entry
-- Create date: 04/12/2025
-- Description: Test the workflow log entry by calling the usp_TrackWorkFlow procedure.
-- ================================================================
EXEC [dbo].[usp_TrackWorkFlow]
    @UserAuthKey = 1,
    @WorkflowStepDescription = N'Test workflow log entry';
GO


Commands completed successfully.

Total execution time: 00:00:00.010

In [84]:
-- 17:
-- ================================================================
-- Author:      Mst Rahi
-- Procedure:   Retrieve recent workflow steps
-- Create date: 04/12/2025
-- Description: Retrieves the top 10 most recent rows from the WorkflowSteps table.
-- ================================================================
SELECT TOP (10) *
FROM [Process].[WorkflowSteps]
ORDER BY WorkFlowStepsKey DESC;
GO


(3 rows affected)

Total execution time: 00:00:00.021

WorkFlowStepDescription,WorkFlowStepTableRowCount,StartingDateTime,EndingDateTime,ClassTime,UserAuthorizationKey,WorkFlowStepsKey
Test workflow log entry,0,2025-04-13 03:14:55.0600000,2025-04-13 03:14:55.0566666,10:45,1,3
Test via stored procedure,0,2025-04-13 03:11:18.4000000,2025-04-13 03:11:18.4000000,10:45,2,2
Test direct insert row,0,2025-04-13 03:11:18.3933333,NULL,10:45,1,1
